## 0. 제출 정보
- 이름: 박재선
- GitHub ID: lilypark0930-lab
- 작성일: 2026.10.04
- 최종 제출 URL: https://github.com/lilypark0930-lab/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문
어떤 상품 카테고리의 완료 주문 기준 금액이 가장 큰가?

### 사용한 파일/컬럼

- order_items.csv : `order_id`, `product_id`, `quantity`, `unit_price`
- products.csv : `product_id`, `category`
- orders.csv : `order_id`, `customer_id`, `order_date`, `order_status`

### 결과 관찰

- 질문에 필요한 정보가 세 파일에 나뉘어 있다. order_items에는 `category`와 `order_status`가 없다.
- 주문 상태는 completed 184건, cancelled 64건, refunded 52건 등 3종류다.
- 주문 기간은 2025-09-15 ~ 2026-09-14다.

### 나의 해석과 판단
왜 이 컬럼과 범위를 선택했는지 작성하세요.
- 금액은 주문 상품 행 단위에서 발생하므로 가장 세부 단위인 order_items에서 계산했다. 주문 단위로 먼저 합치면 한 주문 안의 서로 다른 카테고리를 구분할 수 없다.
- 카테고리는 상품의 속성이므로 products에서 `product_id`로 가져왔다.
- completed만 포함했다.


### 업무·분석적 의미
- 업무 질문 : 완료 주문 기준 매출이 가장 큰 카테고리는?
- 필요한 테이블 : order_items, products, orders
- 필요한 컬럼 : quantity, unit_price, category, order_status, order_date, customer_id
- 연결 키 : order_items–products: `product_id` / order_items–orders: `order_id`
- 포함할 데이터 범위 : `order_status == "completed"` (184건 주문, 474행)
- 계산식 :`line_total = quantity × unit_price`
- 집계 기준 : `category`별 `line_total` 합계
- 검증 기준 : merge 전후 행 수 동일(764), `_merge` 전부 both, 집계 합계 = 원본 completed 합계


### 한계와 추가 확인 사항
- `unit_price`가 정가인지는 확인이 되지 않았다.

Notebook에 자신이 확인하려는 질문과 필요한 데이터를 정리한 화면을 남깁니다.
![질문정리] (images/step01_question.png)
- 각 필요한 파일 데이터 확인 후 관련된 질문 추가한 화면 캡처를 남겼습니다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: `order_status == "completed"`
- 정렬 기준: `total_sales` 내림차순
- 만든 파생 컬럼: `line_total`, `order_month`(`order_date`를 datetime으로 바꾼 뒤 연-월 추출)
- `line_total` 계산식: `order_items["quantity"] * order_items["unit_price"]`

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
- 전체 764행 중 completed는 474행이고, 주문 기준으로는 300건 중 184건이다.
- 상태별 금액: completed 148,990,000원 / cancelled 62,181,000원 / refunded 44,439,000원 (전체 255,610,000원)

### 나의 해석과 판단
필터 기준이 달라지면 결과가 어떻게 달라질지 작성하세요.
- 필터를 적용하지 않으면 전체 금액은 255,610,000원으로, completed 기준보다 약 1.7배 크다.
- 전체 주문 기준으로는 뷰티가 2위(47,551,000원)지만, completed 기준으로는 4위(23,383,000원)로 내려간다. 뷰티는 금액의 50.8%가 취소·환불이기 때문이다.
- 반대로 전자기기는 전체 기준 3위에서 completed 기준 2위로 올라간다.
- 따라서 필터 기준은 "매출의 정의" 자체다.

### 업무·분석적 의미
- 보고서에 어떤 주문 상태를 포함했는지 명시하지 않으면, 같은 데이터로도 다른 부서와 순위가 달라질 수 있다.
- 취소/환불 비율이 높은 카테고리는 별도로 원인을 확인할 가치가 있다.

### 한계와 추가 확인 사항
- 같은 주문 안에서 같은 상품이 두 행으로 나뉜 경우가 10건 있다(예: order_id 49의 product_id 11). 실제로 따로 담은 것인지 중복 입력인지 확인이 필요하다.

## 3. merge 검증
- 병합한 데이터: order_items + products, 그 결과 + orders
- 사용한 key: `product_id`, `order_id`
- `validate` 결과: 두 병합 모두 `many_to_one` 통과 (products의 `product_id`, orders의 `order_id`가 고유함)
- `indicator` 결과: 두 병합 모두 both 764 / left_only 0
- 병합 전/후 행 수: 764행 = 764행 


![merge 검증](images/step03_merge.png)

### 결과 관찰
- 두 병합 모두 validate 오류가 없었고, `_merge`는 전부 both였다.
- 병합 전후 행 수가 같았다.

### 나의 해석과 판단
이번 병합이 안전하다고 판단한 근거를 작성하세요. 행 수가 증가/감소했다면 이유를 설명하세요.
- 오른쪽 테이블의 key가 고유하므로 병합으로 행이 복제되지 않았다.
- left_only가 0이므로 카테고리나 주문 상태가 빠진 행이 없다.
- 행 수와 합계가 모두 같으므로 이번 병합은 데이터를 늘리거나 잃지 않았다고 판단했다.

### 업무·분석적 의미
잘못된 merge가 분석 결과에 어떤 영향을 줄 수 있는지 작성하세요.
- key가 중복되면 행이 복제되어 매출이 실제보다 커지는데, 이 오류는 에러 없이 조용히 생긴다.
- 매칭되지 않은 행은 category가 NaN이 되어 groupby에서 빠지므로 매출이 줄어든다.
- 두 경우 모두 보고 수치를 왜곡하므로 merge 검증은 필수다.

### 한계와 추가 확인 사항
- `validate`는 key의 고유성만 확인한다. key 값 자체가 맞게 입력됐는지는 보장하지 않는다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의: `order_status == "completed"`인 주문 184건의 상세 474행, 총 148,990,000원
- 카테고리별 결과: 1위 스포츠 31,743,000원(21.3%), 2위 전자기기 26,400,000원(17.7%), 3위 생활용품 23,915,000원(16.1%), 최하위 패션 10,587,000원(7.1%)
- 상품별 결과: 1위 스포츠 상품 041 5,705,000원, 2위 식품 상품 012 4,375,000원, 3위 스포츠 상품 009 3,860,000원
- 월별 결과: 최고 2025-12 21,085,000원, 최저 2025-09 2,650,000원
- 고객별 결과: 구매 고객 100명, 상위 10명 합계 34,694,000원(전체의 23.3%), 1위 customer_id 117 4,100,000원

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
수치에서 직접 확인한 상위/하위 또는 변화 패턴을 작성하세요.
- 스포츠가 판매수량(295개)과 주문 수(85건) 모두 1위다.
- 식품은 수량당 평균 금액(약 124,609원)이 가장 높지만 판매수량이 적어 5위다.
- 월별로는 2025-12(21,085,000원), 2026-03(18,323,000원), 2025-10(16,970,000원)이 높고, 2026-02(4,784,000원)와 2026-08(4,365,000원)이 낮아 변동이 크다.

### 나의 해석과 판단
어떤 결과가 가장 중요하다고 판단했는지와 이유를 작성하세요.
- 원래 질문에 직접 답하는 카테고리별 결과가 가장 중요하다.
- 스포츠 1위는 단가가 높아서라기보다 판매수량과 주문 수가 많아서다. 수량당 평균 금액(약 107,603원)은 7개 중 중간 수준이다.
- 2위 전자기기와 4위 뷰티의 차이는 약 300만 원으로 크지 않아서, 기간이나 필터를 바꾸면 순위가 바뀔 수 있다.


### 업무·분석적 의미
다음 의사결정/분석으로 어떻게 연결할지 작성하세요.
- 스포츠는 재고와 마케팅 우선순위를 둘 근거가 된다.
- 다음 분석으로 카테고리별 월 추이와 취소/환불률 시도해볼 수 있을 것이다.


### 한계와 추가 확인 사항
`total_sales`가 회계상 순매출과 같다고 단정할 수 없는 이유를 포함하세요.
- `total_sales`는 `quantity × unit_price`의 단순 합이다. 할인,쿠폰·배송비등 기타 적용 사항이 없고, 주문일 기준으로 집계해 매출 인식 시점(배송완료일 등)도 회계 기준과 다를 수 있다. completed 이후 생기는 반품도 반영되지 않는다. 따라서 회계상 순매출과 같다고 단정할 수 없다.
- 2025-09(15일부터)와 2026-09(14일까지)는 한 달 전체가 아니어서 다른 달과 직접 비교하면 안 된다.
- 표본이 주문 184건으로 작아서, 월별 패턴이나 순위를 일반화하기 어렵다.

## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148,990,000원
- 카테고리 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 고객별 합계: 148,990,000원
- 차이 여부: 차이 없음 (상품별 합계도 148,990,000원)


![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
총합 검증이 필요한 이유와 불일치 시 확인할 순서를 작성하세요.
- 같은 completed 데이터를 기준만 바꿔 나눈 것이라면, 카테고리별·월별·고객별·상품별 합계는 모두 원본 completed `line_total` 합계와 같아야 한다.
- groupby와 merge는 잘못되어도 에러 없이 결과를 낸다. 예를 들어 그룹 기준 값이 NaN인 행은 groupby에서 조용히 빠지고, key가 중복되면 merge 후 행이 복제되어 금액이 부풀려진다. 표만 봐서는 이런 오류를 알아차리기 어렵다.
- 따라서 총합 일치는 "집계 과정에서 데이터가 빠지거나 중복되지 않았다"는 최소한의 증거다.
- 이번 분석에서는 원본 completed 합계와 카테고리·월별·고객별·상품별 합계가 모두 148,990,000원으로 같았다. 따라서 집계 과정에서 누락이나 중복은 없었다고 판단했다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약: LLM이 제안한 카테고리별 매출 코드가 현재 데이터에서 바로 실행되는지 검토하고, `category` 컬럼 위치, 주문 상태 필터링, merge `validate`·`indicator`를 포함한 안전한 수정 코드와 검증 순서를 요청했다
- 제안 코드 요약: `order_items.groupby('category')['line_total'].sum()` → order_items 한 테이블에서 바로 카테고리별 `line_total` 합계를 구하는 코드
- 실제 컬럼/범위와 맞지 않은 부분: 
 - order_items의 실제 컬럼은 `order_item_id, order_id, product_id, quantity, unit_price`뿐이다. `category`는 products.csv에 있어서 그대로 실행하면 `KeyError: 'category'`가 난다.
  - `line_total`은 원본에 없는 컬럼이라 `quantity × unit_price`로 직접 만들어야 한다.
  - 주문 상태 필터가 없다. `order_status`는 orders.csv에 있으므로 병합 후 completed만 남겨야 한다.
  - key 중복으로 행이 복제되거나 매칭에 실패한 행이 빠져도 알 수 없다.
- 수정한 내용:
  1. 세 테이블의 컬럼을 출력해 `category`(products), `order_status`(orders)의 위치를 확인했다.
  2. `line_total = quantity × unit_price`를 생성했다.
  3. `products["product_id"]`, `orders["order_id"]`가 고유한지 `is_unique`로 확인했다.
  4. products, orders를 차례로 `validate="many_to_one"`, `indicator=True`로 병합했다. → 두 병합 모두 validate 통과, `_merge` both 764 / left_only 0
  5. 병합 전후 행 수(764 → 764)와 `line_total` 합계(255,610,000원 → 255,610,000원)가 같은지 `assert`로 확인했다.
  6. `order_status` 값(completed, cancelled, refunded)을 확인한 뒤 completed만 필터링해 카테고리별로 집계했다.
- 최종 판단: 수정 후 사용 

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM 코드가 실행되었다는 것만으로 분석이 맞다고 볼 수 없는 이유를 작성하세요.
- LLM은 실제 데이터 구조를 보지 않고 일반적인 가정으로 코드를 만든다. 이번 코드는 컬럼이 없어 에러가 났지만, 오히려 위험한 건 실행이 되는 경우다.
- 실행 여부가 아니라 컬럼 위치, 분석 범위, 병합 관계, 총합 일치를 직접 확인해야 결과를 믿을 수 있다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 완료 주문 기준 매출 1위는 스포츠(31,743,000원, 21.3%)이며, 판매수량(295개)과 주문 수(85건)도 모두 1위다. 단가보다 판매량이 1위의 원인이다.
2. 주문 상태 필터에 따라 순위가 바뀐다. 뷰티는 전체 기준 2위에서 completed 기준 4위로 내려간다.


### 그 결과를 뒷받침하는 수치/표
- step04 카테고리별 total_sales 표
- step05 총합 검증: 원본 148,990,000원 = 카테고리·월별·고객별 합계 148,990,000원

### 추가로 확인하고 싶은 질문
- 스포츠 매출은 특정 상품(스포츠 상품 041 등)에 집중되어 있는가, 고르게 분포되어 있는가?
- 2026-02와 2026-08의 매출 급감은 주문 수 감소 때문인가, 취소 증가 때문인가?

### 현재 결과의 한계
- 할인,반품,배송비가 반영되지 않은 정가 기준 단순 매출이다.
- completed 주문 184건으로 표본이 작고, 처음과 마지막 달은 한 달 전체가 아니다.
- 같은 주문 안에서 같은 상품이 두 행으로 나뉜 10건이 있어, 중복 입력 여부를 확인하지 못했다.

## 최종 제출 체크
- [O] 핵심 셀 Output이 남아 있습니다.
- [O] merge와 총합 검증 Evidence가 있습니다.
- [O] 결과 관찰과 해석이 구분되어 있습니다.
- [O] LLM 코드를 검증했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [5]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


try:
    PROJECT_ROOT = find_project_root(Path.cwd())
except FileNotFoundError:
    PROJECT_ROOT = Path(r"C:\dev\llm-data-analysis-study")
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)




Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-study\notebooks\ch04
프로젝트 루트: C:\dev\llm-data-analysis-study
데이터 폴더: C:\dev\llm-data-analysis-study\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-study\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [6]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [7]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [8]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [9]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-11
8,9,송지민,M,69,서울,2025-12-26
54,55,윤영철,M,69,광주,2023-10-07
78,79,이서준,M,69,부산,2023-09-20
136,137,김선영,M,68,울산,2025-02-04
45,46,김서현,M,67,대전,2026-07-20
132,133,김성진,M,67,성남,2026-08-05
123,124,김시우,M,67,대구,2024-12-01
11,12,김정남,F,66,대전,2024-12-07
57,58,김성훈,F,66,성남,2025-08-21


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [14]:
products.sort_values(
    "price",
    ascending=False,
).head(10)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


In [15]:


order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [16]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [17]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [18]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [19]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [20]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,11546000,16,721625.0
8,2026-05,16631000,19,875316.0
9,2026-06,12504000,17,735529.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [21]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 13. 결과 저장하기


In [22]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [30]:
category_sales = order_items.groupby('category')['line_total'].sum()

KeyError: 'category'

위와 같이 실행가능하지 않음

##검증 순서
각 테이블의 컬럼 확인 → category, line_total, order_status가 어디 있는지
키 중복 확인 → products.product_id, orders.order_id가 고유한지
병합 시 validate로 관계 강제, indicator로 매칭 안 된 행 확인
병합 전후 행 수와 금액 합계가 같은지 확인
주문 상태 값 확인 후 완료 주문만 필터링
카테고리별 집계

In [32]:
##안전한 수정코드

# 컬럼 위치 확인
print("order_items:", order_items.columns.tolist())
print("products:", products.columns.tolist())
print("orders:", orders.columns.tolist())

# line_total 없으면 생성
if "line_total" not in order_items.columns:
    order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

# 키 중복 확인
assert products["product_id"].is_unique, "products에 중복 product_id"
assert orders["order_id"].is_unique, "orders에 중복 order_id"

# 상품 카테고리 병합 + 검증
merged = order_items.merge(
    products[["product_id", "category"]],
    on="product_id", how="left",
    validate="many_to_one", indicator=True
)
print(merged["_merge"].value_counts())   # left_only가 있으면 상품 정보 누락
merged = merged.drop(columns="_merge")

# 주문 상태 병합 + 검증
merged = merged.merge(
    orders[["order_id", "order_status"]],
    on="order_id", how="left",
    validate="many_to_one", indicator=True
)
print(merged["_merge"].value_counts())   # left_only가 있으면 주문 정보 누락
merged = merged.drop(columns="_merge")

# 행 수·금액 보존 확인 (중복으로 인한 부풀림 방지)
assert len(merged) == len(order_items), "병합 후 행 수가 달라졌습니다"
assert merged["line_total"].sum() == order_items["line_total"].sum()

# 완료 주문만 필터링 (실제 상태 값 먼저 확인)
print(merged["order_status"].unique())
completed = merged[merged["order_status"] == "completed"]

# 카테고리별 매출
category_sales = (
    completed.groupby("category")["line_total"]
    .sum()
    .sort_values(ascending=False)
)
category_sales

order_items: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total', 'amount']
products: ['product_id', 'product_name', 'category', 'price']
orders: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
<ArrowStringArray>
['completed', 'cancelled', 'refunded']
Length: 3, dtype: str


category
스포츠     31743000
전자기기    26400000
생활용품    23915000
뷰티      23383000
식품      16573000
도서      16389000
패션      10587000
Name: line_total, dtype: int64

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [23]:
# 과제 코드를 아래에 작성하세요.
#1. 40세 이상 고객을 추출합니다.
customers_40plus = customers[customers["age"] >= 40]
print(len(customers_40plus), "명")
customers_40plus.head()

78 명


,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-08-19
3,4,조영호,F,55,울산,2026-06-20
6,7,이상현,F,53,인천,2025-02-18
8,9,송지민,M,69,서울,2025-12-26
9,10,유도현,F,62,울산,2024-12-13


In [24]:
#2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
products.sort_values("price").head(10)

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


In [25]:
#3. 결제수단별 주문 수와 비율을 계산합니다.
payment_counts = orders["payment_method"].value_counts()
payment_ratio = orders["payment_method"].value_counts(normalize=True)

payment_summary = pd.DataFrame({
    "주문수": payment_counts,
    "비율(%)": (payment_ratio * 100).round(1)
})
payment_summary

,주문수,비율(%)
payment_method,,
kakao_pay,79,26.3
naver_pay,77,25.7
bank_transfer,74,24.7
card,70,23.3


In [26]:
#4. 카테고리별 평균 상품 가격을 계산합니다.
category_avg_price = (
    products.groupby("category")["price"]
    .mean()
    .round(0)
    .sort_values(ascending=False)
)
category_avg_price

category
식품      137143.0
뷰티      117688.0
패션      115909.0
스포츠     111579.0
도서      106857.0
전자기기    101588.0
생활용품     96438.0
Name: price, dtype: float64

In [27]:
#5. 완료 주문과 전체 주문의 금액 차이
# 주문 상태 값 확인 (완료가 "completed"인지 "완료"인지 등)
print(orders["order_status"].unique())

# 주문 상품별 금액 계산
order_items["amount"] = order_items["quantity"] * order_items["unit_price"]

# 주문 상태 붙이기
items_with_status = order_items.merge(
    orders[["order_id", "order_status"]],
    on="order_id", how="left", validate="many_to_one"
)

total_all = items_with_status["amount"].sum()
total_completed = items_with_status.loc[
    items_with_status["order_status"] == "completed", "amount"
].sum()

print(f"전체 주문 금액: {total_all:,.0f}")
print(f"완료 주문 금액: {total_completed:,.0f}")
print(f"차이: {total_all - total_completed:,.0f}")

<ArrowStringArray>
['completed', 'cancelled', 'refunded']
Length: 3, dtype: str
전체 주문 금액: 255,610,000
완료 주문 금액: 148,990,000
차이: 106,620,000


6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.
pandas로 고객별 구매 금액을 계산하는 코드를 작성해줘.

데이터:
- orders: order_id, customer_id, order_date, order_status
- order_items: order_id, product_id, quantity, unit_price
- customers: customer_id, age 등

요구사항:
1. order_items에 quantity × unit_price로 amount 컬럼을 만든다.
2. order_items와 orders를 order_id로 병합한다 (many_to_one 검증).
3. 고객별 구매 금액 합계를 구하고 customers와 customer_id로 병합한다 (one_to_one 검증).
4. 병합 검증: 병합 전후 행 수 비교, 매칭되지 않은 행 수 확인, 키 중복 여부 확인.
5. 완료 주문만 집계하고, 구매 금액 내림차순으로 정렬해 상위 10명을 출력한다.

In [3]:
## 총합 일치 검증
from pathlib import Path
import pandas as pd


DATA_DIR = Path(r"C:\dev\llm-data-analysis-study") / "data" / "raw"

products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

merged = (
    order_items
    .merge(products[["product_id", "category"]],
           on="product_id", how="left", validate="many_to_one")
    .merge(orders[["order_id", "customer_id", "order_date", "order_status"]],
           on="order_id", how="left", validate="many_to_one")
)

completed = merged[merged["order_status"] == "completed"].copy()
completed["order_month"] = pd.to_datetime(completed["order_date"]).dt.to_period("M")

category_sales = completed.groupby("category")["line_total"].sum()
monthly_sales = completed.groupby("order_month")["line_total"].sum()
customer_sales = completed.groupby("customer_id")["line_total"].sum()
product_sales = completed.groupby("product_id")["line_total"].sum()

original_total = completed["line_total"].sum()

total_check = pd.DataFrame({
    "합계": [
        original_total,
        category_sales.sum(),
        monthly_sales.sum(),
        customer_sales.sum(),
        product_sales.sum(),
    ]
}, index=["원본 completed", "카테고리 합계", "월별 합계", "고객별 합계", "상품별 합계"])

total_check["원본과 차이"] = total_check["합계"] - original_total
print(total_check.map(lambda x: f"{x:,.0f}"))

assert (total_check["원본과 차이"] == 0).all(), "총합이 일치하지 않습니다!"
print("✅ 모든 집계 합계가 원본과 일치합니다.")

                       합계 원본과 차이
원본 completed  148,990,000      0
카테고리 합계       148,990,000      0
월별 합계         148,990,000      0
고객별 합계        148,990,000      0
상품별 합계        148,990,000      0
✅ 모든 집계 합계가 원본과 일치합니다.


## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
